In [1]:
# Importando as bibliotecas

import pandas as pd 
import numpy as np
import os
from datetime import datetime, date
from google.cloud import bigquery
from dotenv import load_dotenv
import pyarrow
import unicodedata



# Carregando variáveis de ambiente
load_dotenv()

True

In [2]:
# Variáveis de ambiente para autenticação e configuração do BigQuery
credencial = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
project_id = os.getenv("PROJECT_ID")

# Tabela SILVER no BigQuery
tabela_silver = os.getenv("SILVER_TABLE")

# Tabela GOLD no BigQuery
# Dim Revendas
tabela_gold_revendas = os.getenv("GOLD_REVENDAS")

#  Dim Distribuidora
tabela_gold_distribuidora = os.getenv("GOLD_DISTRIBUIDORAS")

# Dim Produtos
tabela_gold_produtos = os.getenv("GOLD_PRODUTOS")

# Dim Calendariobject
tabela_gold_calendario = os.getenv("GOLD_CALENDARIO")

# Fato
tabela_gold_fato =  os.getenv("GOLD_FATO")

In [3]:
# Cria cliente BigQuery
client = bigquery.Client.from_service_account_json(credencial, project=project_id)


query = f"""
SELECT *
FROM `{tabela_silver}`
"""

In [4]:
# Executa e converte pra DataFrame
resultado = client.query(query)
df = resultado.to_dataframe()

/home/danielpedro/anaconda3/envs/combustivel_projeto/lib/python3.11/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [5]:
df.head()

,regiao_sigla,estado_sigla,municipio,revenda,cnpj_da_revenda,nome_da_rua,numero_rua,complemento,bairro,cep,produto,data_da_coleta,valor_de_venda,valor_de_compra,unidade_de_medida,bandeira
0,ne,pe,jaboatao dos guararapes,posto dom helder ltda,03792455000185,rua doutor aniceto varejao,s/n,None,candeias,34420-000,diesel s10,2015-01-27,2.640,NaN,r$ / litro,dislub
1,ne,pe,jaboatao dos guararapes,posto dom helder ltda,03792455000185,rua doutor aniceto varejao,s/n,None,candeias,34420-000,diesel s10,2015-02-03,2.848,2.3856,r$ / litro,dislub
2,ne,pe,jaboatao dos guararapes,posto dom helder ltda,03792455000185,rua doutor aniceto varejao,s/n,None,candeias,34420-000,diesel s10,2015-02-10,2.848,NaN,r$ / litro,dislub
3,ne,pe,jaboatao dos guararapes,posto dom helder ltda,03792455000185,rua doutor aniceto varejao,s/n,None,candeias,34420-000,diesel s10,2015-02-18,2.848,NaN,r$ / litro,dislub
4,ne,pe,jaboatao dos guararapes,posto dom helder ltda,03792455000185,rua doutor aniceto varejao,s/n,None,candeias,34420-000,diesel s10,2015-03-16,2.699,2.5406,r$ / litro,dislub


In [ ]:
# Query para criar a tabela fato no GOLD, juntando as dimensões e os fatos da tabela SILVER
query = f"""
CREATE OR REPLACE TABLE `{tabela_gold_fato}` AS
SELECT
    dim_revenda.revenda_sk,
    dim_produto.produto_sk,
    dim_calendario.data_sk,
    dim_distribuidora.bandeira_sk,
    f.valor_de_venda,
    f.valor_de_compra

FROM `{tabela_silver}` f
LEFT JOIN `{tabela_gold_revendas}` dim_revenda
    ON f.cnpj_da_revenda = dim_revenda.cnpj_da_revenda

LEFT JOIN `{tabela_gold_produtos}` dim_produto
    ON f.produto = dim_produto.produto

LEFT JOIN `{tabela_gold_calendario}` dim_calendario
    ON f.data_da_coleta = dim_calendario.data

LEFT JOIN `{tabela_gold_distribuidora}` dim_distribuidora
    ON f.bandeira = dim_distribuidora.bandeira
"""

In [ ]:
# Executa a query para criar a tabela fato e depois faz algumas verificações para garantir que os dados foram inseridos corretamente
try:
    # Executa a query que cria a tabela fato
    job = client.query(query)
    job.result()
    print("✅ Tabela fato criada com sucesso!")
    
    # Conta quantos registros tem na tabela
    check_query = f"SELECT COUNT(*) as total FROM `{tabela_gold_fato}`"
    result = client.query(check_query).result()
    for row in result:
        print(f"Total de registros na fato: {row.total}")
        
    # Pega uma amostra de 5 registros para verificação
    sample_query = f"""
    SELECT 
        revenda_sk,
        produto_sk,
        data_sk,
        bandeira_sk,
        valor_de_venda,
        valor_de_compra
    FROM `{tabela_gold_fato}` 
    LIMIT 5
    """
    sample_result = client.query(sample_query).result()
    
    # Mostra a amostra de dados
    print("\n📋 Amostra dos dados:")
    for i, row in enumerate(sample_result):
        print(f"Registro {i+1}: {dict(row)}")
        
except Exception as e:
    # Se der erro, mostra a mensagem
    print(f"❌ Erro ao criar tabela fato: {e}")

✅ Tabela fato criada com sucesso!
Total de registros na fato: 320556

📋 Amostra dos dados:
Registro 1: {'revenda_sk': 374, 'produto_sk': 4, 'data_sk': 20150114, 'bandeira_sk': 2, 'valor_de_venda': 1.74, 'valor_de_compra': None}
Registro 2: {'revenda_sk': 374, 'produto_sk': 4, 'data_sk': 20150107, 'bandeira_sk': 2, 'valor_de_venda': 1.74, 'valor_de_compra': None}
Registro 3: {'revenda_sk': 384, 'produto_sk': 4, 'data_sk': 20150114, 'bandeira_sk': 3, 'valor_de_venda': 1.74, 'valor_de_compra': None}
Registro 4: {'revenda_sk': 384, 'produto_sk': 4, 'data_sk': 20150107, 'bandeira_sk': 3, 'valor_de_venda': 1.74, 'valor_de_compra': None}
Registro 5: {'revenda_sk': 335, 'produto_sk': 4, 'data_sk': 20150113, 'bandeira_sk': 5, 'valor_de_venda': 1.749, 'valor_de_compra': None}
